In [1]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import os
import glob
import matplotlib.pyplot as plt
import shap
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import r2_score

# ==========================================
# 1. 数据集定义 (缺失值自动填补 + 断层检测 + 全局 Z-score 标准化)
# ==========================================
class TimeAwareMultiStationDataset(Dataset):
    def __init__(self, filepaths, seq_len=24, target_col='GPP_DT_VUT_REF', time_col='date',
                 forcing_cols=None, state_cols=None,
                 static_cols=['Lat', 'Long'], lc_col='Veg_ID',
                 feat_mean_f=None, feat_std_f=None,
                 feat_mean_s=None, feat_std_s=None,
                 static_mean=None, static_std=None,
                 target_mean=None, target_std=None, split_type='train'):

        self.seq_len = seq_len
        self.samples = []

        self.station_forcing = []
        self.station_state = []
        self.station_time_features = []
        self.station_targets = []
        self.station_dates = []

        self.station_static = []
        self.station_lc = []

        for filepath in filepaths:
            data = pd.read_csv(filepath)
            if time_col not in data.columns:
                print(f"⚠️ 在文件 {filepath} 中找不到时间列 '{time_col}'，跳过。")
                continue

            # 异常值清洗：仅替换为 NaN 并直接剔除
            data = data.replace([-9999, -9999.0, -999], np.nan)
            cols_to_clean = forcing_cols + state_cols + [target_col]
            cols_exist = [c for c in cols_to_clean if c in data.columns]

            # 必须保留 dropna：因为 PyTorch 无法计算 NaN，带 NaN 的行必须直接删掉
            data = data.dropna(subset=cols_exist).reset_index(drop=True)

            data[time_col] = pd.to_datetime(data[time_col])
            data = data.sort_values(by=time_col).reset_index(drop=True)

            if len(data) < seq_len:
                continue

            dates = data[time_col]
            self.station_dates.append(dates.values)

            hours = dates.dt.hour.values
            days = dates.dt.dayofyear.values
            time_feats = np.column_stack([
                np.sin(2 * np.pi * hours / 24.0), np.cos(2 * np.pi * hours / 24.0),
                np.sin(2 * np.pi * days / 365.25), np.cos(2 * np.pi * days / 365.25)
            ])

            forcing_data = data[forcing_cols].values
            state_data = data[state_cols].values
            static_data = data[static_cols].values
            lc_data = data[lc_col].values.astype(np.int64)

            if target_col in data.columns:
                targets = data[target_col].values
            else:
                targets = data.iloc[:, -1].values

            self.station_forcing.append(forcing_data)
            self.station_state.append(state_data)
            self.station_time_features.append(time_feats)
            self.station_targets.append(targets)
            self.station_static.append(static_data)
            self.station_lc.append(lc_data)

        if not self.station_forcing:
            raise ValueError(f"加载 {split_type} 数据失败，可能所有数据均被清洗掉或长度不足。")

        all_forcing_concat = np.vstack(self.station_forcing)
        all_state_concat = np.vstack(self.station_state)
        all_static_concat = np.vstack(self.station_static)
        all_targets_concat = np.concatenate(self.station_targets)

        # 特征变量的全局均值与标准差计算，处理除零异常
        self.feat_mean_f = np.mean(all_forcing_concat, axis=0) if feat_mean_f is None else feat_mean_f
        self.feat_std_f = np.std(all_forcing_concat, axis=0) if feat_std_f is None else feat_std_f
        self.feat_std_f[self.feat_std_f == 0] = 1e-8

        self.feat_mean_s = np.mean(all_state_concat, axis=0) if feat_mean_s is None else feat_mean_s
        self.feat_std_s = np.std(all_state_concat, axis=0) if feat_std_s is None else feat_std_s
        self.feat_std_s[self.feat_std_s == 0] = 1e-8

        self.static_mean = np.mean(all_static_concat, axis=0) if static_mean is None else static_mean
        self.static_std = np.std(all_static_concat, axis=0) if static_std is None else static_std
        self.static_std[self.static_std == 0] = 1e-8

        # 目标变量的全局均值与标准差计算
        self.target_mean = np.mean(all_targets_concat) if target_mean is None else target_mean
        self.target_std = np.std(all_targets_concat) if target_std is None else target_std
        if self.target_std == 0:
            self.target_std = 1e-8

        # 全局 Z-score 标准化与样本切分 (包含时间连续性断层检测)
        for i in range(len(self.station_forcing)):
            self.station_forcing[i] = (self.station_forcing[i] - self.feat_mean_f) / self.feat_std_f
            self.station_state[i] = (self.station_state[i] - self.feat_mean_s) / self.feat_std_s
            self.station_static[i] = (self.station_static[i] - self.static_mean) / self.static_std
            self.station_targets[i] = (self.station_targets[i] - self.target_mean) / self.target_std

            dates_array = self.station_dates[i]
            if len(dates_array) > 1:
                diffs = pd.Series(dates_array).diff()
                mode_step = diffs.mode()[0]
                expected_duration = mode_step * (self.seq_len - 1)

                num_samples = len(self.station_forcing[i]) - self.seq_len + 1
                if num_samples > 0:
                    for j in range(num_samples):
                        actual_duration = pd.Timedelta(dates_array[j + self.seq_len - 1] - dates_array[j])
                        if actual_duration == expected_duration:
                            self.samples.append((i, j))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        station_idx, start_idx = self.samples[idx]

        x_forcing = self.station_forcing[station_idx][start_idx : start_idx + self.seq_len]
        x_state = self.station_state[station_idx][start_idx : start_idx + self.seq_len]
        time_x = self.station_time_features[station_idx][start_idx : start_idx + self.seq_len]
        y = self.station_targets[station_idx][start_idx + self.seq_len - 1]
        target_date = self.station_dates[station_idx][start_idx + self.seq_len - 1]

        x_static = self.station_static[station_idx][start_idx : start_idx + self.seq_len]
        x_lc = self.station_lc[station_idx][start_idx : start_idx + self.seq_len]

        return (
            torch.tensor(x_forcing, dtype=torch.float32),
            torch.tensor(x_state, dtype=torch.float32),
            torch.tensor(time_x, dtype=torch.float32),
            torch.tensor(x_static, dtype=torch.float32),
            torch.tensor(x_lc, dtype=torch.long),
            torch.tensor(y, dtype=torch.float32),
            str(target_date)
        )

# ==========================================
# 2. TCN 基础模块定义
# ==========================================
class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super(Chomp1d, self).__init__()
        self.chomp_size = chomp_size
    def forward(self, x):
        return x[:, :, :-self.chomp_size].contiguous()

class TemporalBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernel_size, stride, dilation, padding, dropout=0.2):
        super(TemporalBlock, self).__init__()
        self.conv1 = nn.Conv1d(n_inputs, n_outputs, kernel_size, stride=stride, padding=padding, dilation=dilation)
        self.chomp1 = Chomp1d(padding)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)
        self.conv2 = nn.Conv1d(n_outputs, n_outputs, kernel_size, stride=stride, padding=padding, dilation=dilation)
        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)
        self.net = nn.Sequential(self.conv1, self.chomp1, self.relu1, self.dropout1, self.conv2, self.chomp2, self.relu2, self.dropout2)
        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.ReLU()
    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

class TemporalConvNet(nn.Module):
    def __init__(self, num_inputs, num_channels, kernel_size=3, dropout=0.2):
        super(TemporalConvNet, self).__init__()
        layers = []
        num_levels = len(num_channels)
        for i in range(num_levels):
            dilation_size = 2 ** i
            in_channels = num_inputs if i == 0 else num_channels[i-1]
            out_channels = num_channels[i]
            layers += [TemporalBlock(in_channels, out_channels, kernel_size, stride=1, dilation=dilation_size,
                                     padding=(kernel_size-1) * dilation_size, dropout=dropout)]
        self.network = nn.Sequential(*layers)
    def forward(self, x):
        return self.network(x)

# ==========================================
# 3. 交叉注意力融合模型
# ==========================================
class TCN_Transformer_CrossAttention(nn.Module):
    def __init__(self, num_forcing_features, num_state_features, seq_len,
                 num_static=2, num_lc_classes=13, lc_embed_dim=8,
                 d_model=64, nhead=4, num_layers=2, dim_feedforward=128, dropout=0.1):
        super(TCN_Transformer_CrossAttention, self).__init__()

        self.tcn = TemporalConvNet(num_inputs=num_forcing_features,
                                   num_channels=[d_model] * 6,
                                   kernel_size=3, dropout=dropout)

        self.lc_embedding = nn.Embedding(num_embeddings=num_lc_classes, embedding_dim=lc_embed_dim)

        combined_state_dim = num_state_features + num_static + lc_embed_dim
        self.state_linear = nn.Linear(combined_state_dim, d_model)
        self.time_projector = nn.Linear(4, d_model)

        encoder_layers = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward, dropout=dropout, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers)
        self.cross_attention = nn.MultiheadAttention(embed_dim=d_model, num_heads=nhead, dropout=dropout, batch_first=True)

        self.regressor = nn.Sequential(
            nn.Linear(d_model, d_model // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(d_model // 2, d_model // 4), nn.ReLU(),
            nn.Linear(d_model // 4, 1)
        )

    def forward(self, x_forcing, x_state, time_x, x_static, x_lc):
        x_tcn_in = x_forcing.transpose(1, 2)
        f_tcn = self.tcn(x_tcn_in)
        f_met_memory = f_tcn.transpose(1, 2)

        lc_emb = self.lc_embedding(x_lc)
        combined_state = torch.cat([x_state, x_static, lc_emb], dim=-1)

        x_s_emb = self.state_linear(combined_state)
        time_emb = self.time_projector(time_x)
        x_state_combined = x_s_emb + time_emb
        f_state_global = self.transformer_encoder(x_state_combined)

        fused_features, _ = self.cross_attention(
            query=f_state_global,
            key=f_met_memory,
            value=f_met_memory
        )

        last_step_features = fused_features[:, -1, :]
        out = self.regressor(last_step_features)
        return out.squeeze(-1)

# ==========================================
# 4. SHAP 分析代码
# ==========================================
def perform_shap_analysis(model, dataloader, device, output_img_folder, forcing_cols, state_cols):
    print("\n🔍 开始执行 SHAP 变量重要性分析...")
    model.eval()

    shap_loader = DataLoader(dataloader.dataset, batch_size=4000, shuffle=True)
    batch = next(iter(shap_loader))

    batch_forcing, batch_state, batch_time = batch[0].to(device), batch[1].to(device), batch[2].to(device)
    batch_static, batch_lc = batch[3].to(device), batch[4].to(device)

    bg_size, test_size = 1000, 1000
    if batch_forcing.size(0) < (bg_size + test_size):
        print("⚠️ Batch size 太小，跳过 SHAP 分析。")
        return

    bg_f, bg_s = batch_forcing[:bg_size], batch_state[:bg_size]
    test_f, test_s = batch_forcing[bg_size:bg_size+test_size], batch_state[bg_size:bg_size+test_size]

    class SHAPWrapper(nn.Module):
        def __init__(self, base_model, t_ref, st_ref, lc_ref):
            super().__init__()
            self.base_model = base_model
            self.t_ref = t_ref[:1]
            self.st_ref = st_ref[:1]
            self.lc_ref = lc_ref[:1]

        def forward(self, x_f, x_s):
            b_size = x_f.size(0)
            t_x = self.t_ref.expand(b_size, -1, -1)
            x_st = self.st_ref.expand(b_size, -1, -1)
            x_l = self.lc_ref.expand(b_size, -1)
            out = self.base_model(x_f, x_s, t_x, x_st, x_l)
            return out.unsqueeze(-1)

    wrapper_model = SHAPWrapper(model, batch_time, batch_static, batch_lc).to(device)
    wrapper_model.eval()

    explainer = shap.GradientExplainer(wrapper_model, [bg_f, bg_s])
    shap_values = explainer.shap_values([test_f, test_s])

    if isinstance(shap_values, list) and len(shap_values) == 1 and isinstance(shap_values[0], list):
        shap_values = shap_values[0]

    shap_forcing = np.array(shap_values[0])
    shap_state = np.array(shap_values[1])

    shap_forcing_2d = shap_forcing.reshape(-1, len(forcing_cols))
    shap_state_2d = shap_state.reshape(-1, len(state_cols))

    test_f_2d = test_f.cpu().numpy().reshape(-1, len(forcing_cols))
    test_s_2d = test_s.cpu().numpy().reshape(-1, len(state_cols))

    shap_combined = np.concatenate([shap_forcing_2d, shap_state_2d], axis=1)
    features_combined = np.concatenate([test_f_2d, test_s_2d], axis=1)
    feature_names = forcing_cols + state_cols

    plt.figure(figsize=(12, 10))
    shap.summary_plot(shap_combined, features_combined, feature_names=feature_names, show=False)
    plt.title("SHAP Summary: Global Feature Impact on GPP (Time-Flattened)", fontname='Arial', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(output_img_folder, "SHAP_Summary_Plot.png"), dpi=300)
    plt.close()

    plt.figure(figsize=(12, 10))
    shap.summary_plot(shap_combined, features_combined, feature_names=feature_names, plot_type="bar", show=False)
    plt.title("SHAP Global Feature Importance (Magnitude)", fontname='Arial', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(output_img_folder, "SHAP_Bar_Plot.png"), dpi=300)
    plt.close()

    print(f"✅ SHAP 生成完成，保存至: {output_img_folder}")

# ==========================================
# 5. 主流程
# ==========================================
def train_time_aware_model():
    seq_len = 96
    batch_size = 64
    num_epochs = 100
    learning_rate = 0.001
    patience = 10
    TIME_COLUMN_NAME = 'date'
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    data_folder = r"C:\Users\Admin\Desktop\实验四\全波段全变量DT"
    output_img_folder = os.path.join(data_folder, "全149站点全局标准化单农田")
    os.makedirs(output_img_folder, exist_ok=True)

    forcing_cols = ['SW_IN_F', 'SW_IN_POT', 'CO2_F_MDS', 'P_F', 'VPD_F', 'TA_F', 'TS_F_MDS_1', 'SWC_F_MDS_1', 'WS_F']
    state_cols = ['EPIC_Available_Mask', 'Band317nm_Ref', 'Band325nm_Ref', 'Band340nm_Ref',
                  'Band388nm_Ref', 'Band443nm_Ref', 'Band551nm_Ref', 'Band680nm_Ref',
                  'Band688nm_Ref', 'Band764nm_Ref', 'Band780nm_Ref', 'Mean_SZA', 'Mean_VZA', 'Mean_RAA']
    static_cols = ['Lat', 'Long']
    lc_col = 'Veg_ID'

    # ==========================================
    # 在这里指定你的 训练集、验证集、测试集 站点名称
    # 只要 CSV 文件名包含列表中的字符串就会被划分入对应集合
    # ==========================================
    train_sites = [
        'AU-MvB_CRO', 'CH-Oe2_CRO', 'CH-Tnk_CRO', 'CN-Dmn_CRO',
        'CN-Jzh_CRO', 'CN-Yuc_CRO', 'CR-Fsc_CRO', 'CZ-KrP_CRO',
        'DE-RuM_CRO', 'DE-RuS_CRO', 'FR-Aur_CRO', 'KR-Hnm_CRO',
        'US-A39_CRO'
        # <-- 来自CSV表中的其余CRO站点分配为训练集
    ]

    val_sites = [
        'US-Bi2_CRO', 'US-Mo1_CRO', 'US-Ne1_CRO', 'US-Ne2_CRO',
        'US-Ro1_CRO', 'US-xSL_CRO'
        # <-- 来自CSV表中的其余CRO站点分配为验证集
    ]

    test_sites = [
        'US-Ne3_CRO', 'KR-CRK_CRO', 'CN-Nmn_CRO', 'BE-Lon_CRO'
        # <-- 仅保留您指定的四个测试站点
    ]

    all_files = glob.glob(os.path.join(data_folder, "*.[cC][sS][vV]"))
    if not all_files:
        print("❌ 错误：未在指定目录找到CSV文件。")
        return

    # 严格按照给定的站点列表划分文件
    train_files, val_files, test_files = [], [], []
    unused_files = []

    for f in all_files:
        fname = os.path.basename(f)
        if any(site in fname for site in train_sites):
            train_files.append(f)
        elif any(site in fname for site in val_sites):
            val_files.append(f)
        elif any(site in fname for site in test_sites):
            test_files.append(f)
        else:
            unused_files.append(fname)

    print(f"📁 文件夹中共找到 {len(all_files)} 个站点文件。")
    print(f"   -> 划分为训练集: {len(train_files)} | 验证集: {len(val_files)} | 测试集: {len(test_files)}")

    if unused_files:
        print(f"⚠️ 注意: 有 {len(unused_files)} 个文件未被划分到任何集合，将被忽略。")
        # print("未分配的文件列表: ", unused_files) # 取消注释以查看未分配的文件名

    if not train_files:
        print("❌ 错误：训练集文件为空，请检查 train_sites 列表是否正确匹配了文件名。")
        return

    train_dataset = TimeAwareMultiStationDataset(
        train_files, seq_len, time_col=TIME_COLUMN_NAME,
        forcing_cols=forcing_cols, state_cols=state_cols,
        static_cols=static_cols, lc_col=lc_col, split_type='train'
    )

    # 提取训练集计算出的输入特征与目标变量的全局均值和标准差
    f_mean_f, f_std_f = train_dataset.feat_mean_f, train_dataset.feat_std_f
    f_mean_s, f_std_s = train_dataset.feat_mean_s, train_dataset.feat_std_s
    s_mean, s_std = train_dataset.static_mean, train_dataset.static_std
    t_mean, t_std = train_dataset.target_mean, train_dataset.target_std

    val_dataset = TimeAwareMultiStationDataset(
        val_files, seq_len, time_col=TIME_COLUMN_NAME,
        forcing_cols=forcing_cols, state_cols=state_cols,
        static_cols=static_cols, lc_col=lc_col,
        feat_mean_f=f_mean_f, feat_std_f=f_std_f,
        feat_mean_s=f_mean_s, feat_std_s=f_std_s,
        static_mean=s_mean, static_std=s_std,
        target_mean=t_mean, target_std=t_std, split_type='val'
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    model = TCN_Transformer_CrossAttention(
        num_forcing_features=len(forcing_cols),
        num_state_features=len(state_cols),
        seq_len=seq_len,
        num_static=len(static_cols),
        num_lc_classes=13,
        lc_embed_dim=8
    ).to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    checkpoint_latest_path = os.path.join(output_img_folder, "checkpoint_latest.pth")
    checkpoint_best_path = os.path.join(output_img_folder, "checkpoint_best.pth")

    start_epoch = 0
    best_val_loss = float('inf')
    epochs_no_improve = 0

    if os.path.exists(checkpoint_latest_path):
        print(f"\n🔄 检测到本地进度，尝试恢复...")
        try:
            checkpoint = torch.load(checkpoint_latest_path, map_location=device)
            model.load_state_dict(checkpoint['model_state_dict'])
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            start_epoch = checkpoint['epoch'] + 1
            best_val_loss = checkpoint['best_val_loss']
            epochs_no_improve = checkpoint['epochs_no_improve']
            print(f"✅ 从第 {start_epoch} 轮恢复，历史最佳 MSE: {best_val_loss:.4f}")
        except Exception as e:
            print(f"⚠️ 无法加载历史进度，将重新开始。错误信息: {e}")

    if start_epoch < num_epochs and epochs_no_improve < patience:
        print(f"🚀 开始训练...\n" + "-"*40)
        for epoch in range(start_epoch, num_epochs):
            model.train()
            train_loss = 0

            for batch_forcing, batch_state, batch_time, batch_static, batch_lc, batch_y, _ in train_loader:
                batch_forcing, batch_state, batch_time = batch_forcing.to(device), batch_state.to(device), batch_time.to(device)
                batch_static, batch_lc, batch_y = batch_static.to(device), batch_lc.to(device), batch_y.to(device)

                optimizer.zero_grad()
                outputs = model(batch_forcing, batch_state, batch_time, batch_static, batch_lc)
                loss = criterion(outputs, batch_y)
                loss.backward()

                # 梯度裁剪
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

                optimizer.step()
                train_loss += loss.item()

            model.eval()
            val_loss = 0
            with torch.no_grad():
                for batch_forcing, batch_state, batch_time, batch_static, batch_lc, batch_y, _ in val_loader:
                    batch_forcing, batch_state, batch_time = batch_forcing.to(device), batch_state.to(device), batch_time.to(device)
                    batch_static, batch_lc, batch_y = batch_static.to(device), batch_lc.to(device), batch_y.to(device)

                    outputs = model(batch_forcing, batch_state, batch_time, batch_static, batch_lc)
                    loss = criterion(outputs, batch_y)
                    val_loss += loss.item()

            avg_train_loss = train_loss / len(train_loader)
            avg_val_loss = val_loss / len(val_loader)
            print(f"Epoch [{epoch+1:03d}/{num_epochs}] | Train MSE (Std Scaled): {avg_train_loss:.4f} | Val MSE (Std Scaled): {avg_val_loss:.4f}")

            if avg_val_loss < best_val_loss:
                print(f"   🌟 新的最佳模型！MSE: {avg_val_loss:.4f}。")
                best_val_loss = avg_val_loss
                epochs_no_improve = 0
                torch.save(model.state_dict(), checkpoint_best_path)
            else:
                epochs_no_improve += 1
                print(f"   ⏳ 验证集未改善 (早停: {epochs_no_improve}/{patience})")

            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_val_loss': best_val_loss,
                'epochs_no_improve': epochs_no_improve
            }, checkpoint_latest_path)

            if epochs_no_improve >= patience:
                print(f"\n🛑 早停机制触发，训练结束。")
                break
            print("-" * 40)

    # ==========================================
    # 6. 分站点独立测试评估 (整合高级绘图逻辑)
    # ==========================================
    if os.path.exists(checkpoint_best_path):
        print(f"\n🎯 开始测试集评估...")
        model.load_state_dict(torch.load(checkpoint_best_path, map_location=device))
    model.eval()

    global_all_preds, global_all_targets = [], []
    for test_file in test_files:
        station_name = os.path.basename(test_file).replace('.csv', '')
        single_test_dataset = TimeAwareMultiStationDataset(
            [test_file], seq_len, time_col=TIME_COLUMN_NAME,
            forcing_cols=forcing_cols, state_cols=state_cols,
            static_cols=static_cols, lc_col=lc_col,
            feat_mean_f=f_mean_f, feat_std_f=f_std_f,
            feat_mean_s=f_mean_s, feat_std_s=f_std_s,
            static_mean=s_mean, static_std=s_std,
            target_mean=t_mean, target_std=t_std, split_type='test'
        )

        if len(single_test_dataset) == 0: continue

        single_test_loader = DataLoader(single_test_dataset, batch_size=batch_size, shuffle=False)
        all_preds, all_targets, all_times = [], [], []

        with torch.no_grad():
            for batch_forcing, batch_state, batch_time, batch_static, batch_lc, batch_y, batch_dt in single_test_loader:
                batch_forcing, batch_state, batch_time = batch_forcing.to(device), batch_state.to(device), batch_time.to(device)
                batch_static, batch_lc = batch_static.to(device), batch_lc.to(device)

                outputs = model(batch_forcing, batch_state, batch_time, batch_static, batch_lc)
                all_preds.extend(outputs.cpu().numpy())
                all_targets.extend(batch_y.numpy())
                all_times.extend(batch_dt)

        all_preds, all_targets = np.array(all_preds), np.array(all_targets)
        all_times = pd.to_datetime(all_times)

        # 反标准化：将模型输出的 Z-score 还原回原始的 GPP 物理刻度
        all_preds = all_preds * t_std + t_mean
        all_targets = all_targets * t_std + t_mean

        # 应用数据清洗要求：剔除 GPP < 0 的异常值
        valid_mask = all_targets >= 0
        plot_targets = all_targets[valid_mask]
        plot_preds = all_preds[valid_mask]
        plot_times = all_times[valid_mask]

        if len(plot_targets) == 0:
            continue

        global_all_preds.extend(plot_preds)
        global_all_targets.extend(plot_targets)

        station_mse = np.mean((plot_preds - plot_targets)**2)
        station_r2 = r2_score(plot_targets, plot_preds)

        print(f"📢 站点: {station_name} | 测试集 MSE: {station_mse:.4f} | 测试集 R²: {station_r2:.4f}")

        # 图表 1: 滑动平均趋势图
        window_size = 12
        all_targets_smooth = pd.Series(plot_targets).rolling(window=window_size, min_periods=1).mean().values
        all_preds_smooth = pd.Series(plot_preds).rolling(window=window_size, min_periods=1).mean().values

        plt.figure(figsize=(15, 6))
        plt.plot(plot_times, plot_targets, color='royalblue', linewidth=0.5, alpha=0.2, label='Actual (Raw)')
        plt.plot(plot_times, plot_preds, color='crimson', linewidth=0.5, alpha=0.2, label='Predicted (Raw)')
        plt.plot(plot_times, all_targets_smooth, label=f'Actual GPP (MA-{window_size})', color='royalblue', linewidth=1.5, alpha=0.9)
        plt.plot(plot_times, all_preds_smooth, label=f'Predicted GPP (MA-{window_size})', color='crimson', linewidth=1.5, linestyle='--', alpha=0.9)

        plt.title(f'[{station_name}] Test Set GPP Prediction (Moving Average Window={window_size})', fontsize=14, fontname='Arial')
        plt.xlabel('Time', fontsize=12, fontname='Arial')
        plt.ylabel('GPP Value', fontsize=12, fontname='Arial')
        plt.legend(prop={'family': 'Arial'})

        ax = plt.gca()
        ax.tick_params(direction='in')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        plt.grid(True, which='both', linestyle='--', alpha=0.5)
        plt.tight_layout()
        plt.savefig(os.path.join(output_img_folder, f"{station_name}_trend_ma.png"), dpi=300, facecolor='white', transparent=False)
        plt.close()

        # 图表 2: 局部放大图 (30天)
        zoom_days = 30
        steps_per_day = 24
        zoom_steps = zoom_days * steps_per_day
        zoom_steps = min(zoom_steps, len(plot_times))

        if zoom_steps > 0:
            peak_idx = np.argmax(all_targets_smooth)
            start_idx = max(0, peak_idx - zoom_steps // 2)
            end_idx = min(len(plot_times), start_idx + zoom_steps)

            if end_idx - start_idx < zoom_steps:
                start_idx = max(0, end_idx - zoom_steps)

            plt.figure(figsize=(15, 5))
            plt.plot(plot_times[start_idx:end_idx], plot_targets[start_idx:end_idx],
                     label='Actual GPP (Raw)', color='royalblue', linewidth=1.5)
            plt.plot(plot_times[start_idx:end_idx], plot_preds[start_idx:end_idx],
                     label='Predicted GPP (Raw)', color='crimson', linewidth=1.5, linestyle='--', alpha=0.8)

            peak_date_str = pd.to_datetime(plot_times.iloc[peak_idx] if isinstance(plot_times, pd.Series) else plot_times[peak_idx]).strftime('%Y-%m-%d')

            plt.title(f'[{station_name}] 30-Day Zoomed Prediction (Raw Data Detail around {peak_date_str})', fontsize=14, fontname='Arial')
            plt.xlabel('Time', fontsize=12, fontname='Arial')
            plt.ylabel('GPP Value', fontsize=12, fontname='Arial')
            plt.legend(prop={'family': 'Arial'})

            ax = plt.gca()
            ax.tick_params(direction='in')
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            plt.grid(True, which='both', linestyle='--', alpha=0.6)
            plt.xticks(rotation=30)
            plt.tight_layout()
            plt.savefig(os.path.join(output_img_folder, f"{station_name}_zoom_30days.png"), dpi=300, facecolor='white', transparent=False)
            plt.close()

        # 图表 3: 散点图
        plt.figure(figsize=(6, 6))
        plt.scatter(plot_targets, plot_preds, alpha=0.6, color='teal', s=15, edgecolors='k', linewidth=0.2)

        min_val = min(plot_targets.min(), plot_preds.min())
        max_val = max(plot_targets.max(), plot_preds.max())
        plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='1:1 Line')

        plt.title(f'[{station_name}] Actual vs Predicted Scatter', fontname='Arial')
        plt.xlabel('Actual GPP', fontname='Arial')
        plt.ylabel('Predicted GPP', fontname='Arial')
        plt.legend(prop={'family': 'Arial'})

        ax = plt.gca()
        ax.tick_params(direction='in')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        plt.tight_layout()
        plt.savefig(os.path.join(output_img_folder, f"{station_name}_scatter.png"), dpi=300, facecolor='white', transparent=False)
        plt.close()

        # ==========================================
        # 图表 4: 抽取任一年的数据展示时间序列
        # ==========================================
        years = plot_times.dt.year if hasattr(plot_times, 'dt') else pd.Series(plot_times).dt.year
        unique_years = years.unique()

        if len(unique_years) > 0:
            target_year = years.value_counts().idxmax()
            year_mask = (years == target_year).values

            y_times = plot_times[year_mask]
            y_targets = plot_targets[year_mask]
            y_preds = plot_preds[year_mask]

            plt.figure(figsize=(15, 5))
            plt.plot(y_times, y_targets, label=f'Actual GPP ({target_year})', color='royalblue', linewidth=1.2, alpha=0.8)
            plt.plot(y_times, y_preds, label=f'Predicted GPP ({target_year})', color='crimson', linewidth=1.2, linestyle='--', alpha=0.8)

            plt.title(f'[{station_name}] Full Year GPP Time Series ({target_year})', fontsize=14, fontname='Arial')
            plt.xlabel('Time', fontsize=12, fontname='Arial')
            plt.ylabel('GPP Value', fontsize=12, fontname='Arial')
            plt.legend(prop={'family': 'Arial'}, loc='upper right')

            ax = plt.gca()
            ax.tick_params(direction='in')
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            plt.grid(True, which='both', linestyle='--', alpha=0.4)
            plt.tight_layout()
            plt.savefig(os.path.join(output_img_folder, f"{station_name}_singleyear_{target_year}.png"), dpi=300, facecolor='white', transparent=False)
            plt.close()

    # ------------------------------------------
    # 7. 全局指标汇总与 SHAP 分析
    # ------------------------------------------
    if len(global_all_targets) > 0:
        global_all_preds = np.array(global_all_preds)
        global_all_targets = np.array(global_all_targets)

        global_mse = np.mean((global_all_preds - global_all_targets)**2)
        global_r2 = r2_score(global_all_targets, global_all_preds)

        print("\n" + "="*50)
        print("🌎 所有站点测试集全局评估结果")
        print("="*50)
        print(f"有效总测试样本数 (剔除 GPP<0): {len(global_all_targets)}")
        print(f"Global Test MSE: {global_mse:.4f}")
        print(f"Global Test R² Score: {global_r2:.4f}")
        print("="*50)

        try:
            perform_shap_analysis(model, val_loader, device, output_img_folder, forcing_cols, state_cols)
        except Exception as e:
            print(f"⚠️ SHAP 分析生成失败: {e}")

if __name__ == "__main__":
    train_time_aware_model()

D:\miniconda3\envs\dl_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📁 文件夹中共找到 149 个站点文件。
   -> 划分为训练集: 13 | 验证集: 6 | 测试集: 4
⚠️ 注意: 有 126 个文件未被划分到任何集合，将被忽略。
🚀 开始训练...
----------------------------------------
Epoch [001/100] | Train MSE (Std Scaled): 0.2552 | Val MSE (Std Scaled): 0.8165
   🌟 新的最佳模型！MSE: 0.8165。
----------------------------------------
Epoch [002/100] | Train MSE (Std Scaled): 0.1427 | Val MSE (Std Scaled): 0.6858
   🌟 新的最佳模型！MSE: 0.6858。
----------------------------------------
Epoch [003/100] | Train MSE (Std Scaled): 0.0990 | Val MSE (Std Scaled): 0.5534
   🌟 新的最佳模型！MSE: 0.5534。
----------------------------------------
Epoch [004/100] | Train MSE (Std Scaled): 0.0787 | Val MSE (Std Scaled): 0.6270
   ⏳ 验证集未改善 (早停: 1/10)
----------------------------------------
Epoch [005/100] | Train MSE (Std Scaled): 0.0668 | Val MSE (Std Scaled): 0.6641
   ⏳ 验证集未改善 (早停: 2/10)
----------------------------------------
Epoch [006/100] | Train MSE (Std Scaled): 0.0588 | Val MSE (Std Scaled): 0.6278
   ⏳ 验证集未改善 (早停: 3/10)
--------------------------------

C:\Users\admin\AppData\Local\Temp\ipykernel_10108\985094103.py:311: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_combined, features_combined, feature_names=feature_names, show=False)
C:\Users\admin\AppData\Local\Temp\ipykernel_10108\985094103.py:318: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_combined, features_combined, feature_names=feature_names, plot_type="bar", show=False)


✅ SHAP 生成完成，保存至: C:\Users\Admin\Desktop\实验四\全波段全变量DT\全149站点全局标准化单农田


In [5]:
#森林
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import os
import glob
import matplotlib.pyplot as plt
import shap
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import r2_score

# ==========================================
# 1. 数据集定义 (缺失值自动填补 + 断层检测 + 全局 Z-score 标准化)
# ==========================================
class TimeAwareMultiStationDataset(Dataset):
    def __init__(self, filepaths, seq_len=24, target_col='GPP_DT_VUT_REF', time_col='date',
                 forcing_cols=None, state_cols=None,
                 static_cols=['Lat', 'Long'], lc_col='Veg_ID',
                 feat_mean_f=None, feat_std_f=None,
                 feat_mean_s=None, feat_std_s=None,
                 static_mean=None, static_std=None,
                 target_mean=None, target_std=None, split_type='train'):

        self.seq_len = seq_len
        self.samples = []

        self.station_forcing = []
        self.station_state = []
        self.station_time_features = []
        self.station_targets = []
        self.station_dates = []

        self.station_static = []
        self.station_lc = []

        for filepath in filepaths:
            data = pd.read_csv(filepath)
            if time_col not in data.columns:
                print(f"⚠️ 在文件 {filepath} 中找不到时间列 '{time_col}'，跳过。")
                continue

            # 异常值清洗：仅替换为 NaN 并直接剔除
            data = data.replace([-9999, -9999.0, -999], np.nan)
            cols_to_clean = forcing_cols + state_cols + [target_col]
            cols_exist = [c for c in cols_to_clean if c in data.columns]

            # 必须保留 dropna：因为 PyTorch 无法计算 NaN，带 NaN 的行必须直接删掉
            data = data.dropna(subset=cols_exist).reset_index(drop=True)

            data[time_col] = pd.to_datetime(data[time_col])
            data = data.sort_values(by=time_col).reset_index(drop=True)

            if len(data) < seq_len:
                continue

            dates = data[time_col]
            self.station_dates.append(dates.values)

            hours = dates.dt.hour.values
            days = dates.dt.dayofyear.values
            time_feats = np.column_stack([
                np.sin(2 * np.pi * hours / 24.0), np.cos(2 * np.pi * hours / 24.0),
                np.sin(2 * np.pi * days / 365.25), np.cos(2 * np.pi * days / 365.25)
            ])

            forcing_data = data[forcing_cols].values
            state_data = data[state_cols].values
            static_data = data[static_cols].values
            lc_data = data[lc_col].values.astype(np.int64)

            if target_col in data.columns:
                targets = data[target_col].values
            else:
                targets = data.iloc[:, -1].values

            self.station_forcing.append(forcing_data)
            self.station_state.append(state_data)
            self.station_time_features.append(time_feats)
            self.station_targets.append(targets)
            self.station_static.append(static_data)
            self.station_lc.append(lc_data)

        if not self.station_forcing:
            raise ValueError(f"加载 {split_type} 数据失败，可能所有数据均被清洗掉或长度不足。")

        all_forcing_concat = np.vstack(self.station_forcing)
        all_state_concat = np.vstack(self.station_state)
        all_static_concat = np.vstack(self.station_static)
        all_targets_concat = np.concatenate(self.station_targets)

        # 特征变量的全局均值与标准差计算，处理除零异常
        self.feat_mean_f = np.mean(all_forcing_concat, axis=0) if feat_mean_f is None else feat_mean_f
        self.feat_std_f = np.std(all_forcing_concat, axis=0) if feat_std_f is None else feat_std_f
        self.feat_std_f[self.feat_std_f == 0] = 1e-8

        self.feat_mean_s = np.mean(all_state_concat, axis=0) if feat_mean_s is None else feat_mean_s
        self.feat_std_s = np.std(all_state_concat, axis=0) if feat_std_s is None else feat_std_s
        self.feat_std_s[self.feat_std_s == 0] = 1e-8

        self.static_mean = np.mean(all_static_concat, axis=0) if static_mean is None else static_mean
        self.static_std = np.std(all_static_concat, axis=0) if static_std is None else static_std
        self.static_std[self.static_std == 0] = 1e-8

        # 目标变量的全局均值与标准差计算
        self.target_mean = np.mean(all_targets_concat) if target_mean is None else target_mean
        self.target_std = np.std(all_targets_concat) if target_std is None else target_std
        if self.target_std == 0:
            self.target_std = 1e-8

        # 全局 Z-score 标准化与样本切分 (包含时间连续性断层检测)
        for i in range(len(self.station_forcing)):
            self.station_forcing[i] = (self.station_forcing[i] - self.feat_mean_f) / self.feat_std_f
            self.station_state[i] = (self.station_state[i] - self.feat_mean_s) / self.feat_std_s
            self.station_static[i] = (self.station_static[i] - self.static_mean) / self.static_std
            self.station_targets[i] = (self.station_targets[i] - self.target_mean) / self.target_std

            dates_array = self.station_dates[i]
            if len(dates_array) > 1:
                diffs = pd.Series(dates_array).diff()
                mode_step = diffs.mode()[0]
                expected_duration = mode_step * (self.seq_len - 1)

                num_samples = len(self.station_forcing[i]) - self.seq_len + 1
                if num_samples > 0:
                    for j in range(num_samples):
                        actual_duration = pd.Timedelta(dates_array[j + self.seq_len - 1] - dates_array[j])
                        if actual_duration == expected_duration:
                            self.samples.append((i, j))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        station_idx, start_idx = self.samples[idx]

        x_forcing = self.station_forcing[station_idx][start_idx : start_idx + self.seq_len]
        x_state = self.station_state[station_idx][start_idx : start_idx + self.seq_len]
        time_x = self.station_time_features[station_idx][start_idx : start_idx + self.seq_len]
        y = self.station_targets[station_idx][start_idx + self.seq_len - 1]
        target_date = self.station_dates[station_idx][start_idx + self.seq_len - 1]

        x_static = self.station_static[station_idx][start_idx : start_idx + self.seq_len]
        x_lc = self.station_lc[station_idx][start_idx : start_idx + self.seq_len]

        return (
            torch.tensor(x_forcing, dtype=torch.float32),
            torch.tensor(x_state, dtype=torch.float32),
            torch.tensor(time_x, dtype=torch.float32),
            torch.tensor(x_static, dtype=torch.float32),
            torch.tensor(x_lc, dtype=torch.long),
            torch.tensor(y, dtype=torch.float32),
            str(target_date)
        )

# ==========================================
# 2. TCN 基础模块定义
# ==========================================
class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super(Chomp1d, self).__init__()
        self.chomp_size = chomp_size
    def forward(self, x):
        return x[:, :, :-self.chomp_size].contiguous()

class TemporalBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernel_size, stride, dilation, padding, dropout=0.2):
        super(TemporalBlock, self).__init__()
        self.conv1 = nn.Conv1d(n_inputs, n_outputs, kernel_size, stride=stride, padding=padding, dilation=dilation)
        self.chomp1 = Chomp1d(padding)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)
        self.conv2 = nn.Conv1d(n_outputs, n_outputs, kernel_size, stride=stride, padding=padding, dilation=dilation)
        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)
        self.net = nn.Sequential(self.conv1, self.chomp1, self.relu1, self.dropout1, self.conv2, self.chomp2, self.relu2, self.dropout2)
        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.ReLU()
    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

class TemporalConvNet(nn.Module):
    def __init__(self, num_inputs, num_channels, kernel_size=3, dropout=0.2):
        super(TemporalConvNet, self).__init__()
        layers = []
        num_levels = len(num_channels)
        for i in range(num_levels):
            dilation_size = 2 ** i
            in_channels = num_inputs if i == 0 else num_channels[i-1]
            out_channels = num_channels[i]
            layers += [TemporalBlock(in_channels, out_channels, kernel_size, stride=1, dilation=dilation_size,
                                     padding=(kernel_size-1) * dilation_size, dropout=dropout)]
        self.network = nn.Sequential(*layers)
    def forward(self, x):
        return self.network(x)

# ==========================================
# 3. 交叉注意力融合模型
# ==========================================
class TCN_Transformer_CrossAttention(nn.Module):
    def __init__(self, num_forcing_features, num_state_features, seq_len,
                 num_static=2, num_lc_classes=13, lc_embed_dim=8,
                 d_model=64, nhead=4, num_layers=2, dim_feedforward=128, dropout=0.1):
        super(TCN_Transformer_CrossAttention, self).__init__()

        self.tcn = TemporalConvNet(num_inputs=num_forcing_features,
                                   num_channels=[d_model] * 6,
                                   kernel_size=3, dropout=dropout)

        self.lc_embedding = nn.Embedding(num_embeddings=num_lc_classes, embedding_dim=lc_embed_dim)

        combined_state_dim = num_state_features + num_static + lc_embed_dim
        self.state_linear = nn.Linear(combined_state_dim, d_model)
        self.time_projector = nn.Linear(4, d_model)

        encoder_layers = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward, dropout=dropout, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers)
        self.cross_attention = nn.MultiheadAttention(embed_dim=d_model, num_heads=nhead, dropout=dropout, batch_first=True)

        self.regressor = nn.Sequential(
            nn.Linear(d_model, d_model // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(d_model // 2, d_model // 4), nn.ReLU(),
            nn.Linear(d_model // 4, 1)
        )

    def forward(self, x_forcing, x_state, time_x, x_static, x_lc):
        x_tcn_in = x_forcing.transpose(1, 2)
        f_tcn = self.tcn(x_tcn_in)
        f_met_memory = f_tcn.transpose(1, 2)

        lc_emb = self.lc_embedding(x_lc)
        combined_state = torch.cat([x_state, x_static, lc_emb], dim=-1)

        x_s_emb = self.state_linear(combined_state)
        time_emb = self.time_projector(time_x)
        x_state_combined = x_s_emb + time_emb
        f_state_global = self.transformer_encoder(x_state_combined)

        fused_features, _ = self.cross_attention(
            query=f_state_global,
            key=f_met_memory,
            value=f_met_memory
        )

        last_step_features = fused_features[:, -1, :]
        out = self.regressor(last_step_features)
        return out.squeeze(-1)

# ==========================================
# 4. SHAP 分析代码
# ==========================================
def perform_shap_analysis(model, dataloader, device, output_img_folder, forcing_cols, state_cols):
    print("\n🔍 开始执行 SHAP 变量重要性分析...")
    model.eval()

    shap_loader = DataLoader(dataloader.dataset, batch_size=4000, shuffle=True)
    batch = next(iter(shap_loader))

    batch_forcing, batch_state, batch_time = batch[0].to(device), batch[1].to(device), batch[2].to(device)
    batch_static, batch_lc = batch[3].to(device), batch[4].to(device)

    bg_size, test_size = 1000, 1000
    if batch_forcing.size(0) < (bg_size + test_size):
        print("⚠️ Batch size 太小，跳过 SHAP 分析。")
        return

    bg_f, bg_s = batch_forcing[:bg_size], batch_state[:bg_size]
    test_f, test_s = batch_forcing[bg_size:bg_size+test_size], batch_state[bg_size:bg_size+test_size]

    class SHAPWrapper(nn.Module):
        def __init__(self, base_model, t_ref, st_ref, lc_ref):
            super().__init__()
            self.base_model = base_model
            self.t_ref = t_ref[:1]
            self.st_ref = st_ref[:1]
            self.lc_ref = lc_ref[:1]

        def forward(self, x_f, x_s):
            b_size = x_f.size(0)
            t_x = self.t_ref.expand(b_size, -1, -1)
            x_st = self.st_ref.expand(b_size, -1, -1)
            x_l = self.lc_ref.expand(b_size, -1)
            out = self.base_model(x_f, x_s, t_x, x_st, x_l)
            return out.unsqueeze(-1)

    wrapper_model = SHAPWrapper(model, batch_time, batch_static, batch_lc).to(device)
    wrapper_model.eval()

    explainer = shap.GradientExplainer(wrapper_model, [bg_f, bg_s])
    shap_values = explainer.shap_values([test_f, test_s])

    if isinstance(shap_values, list) and len(shap_values) == 1 and isinstance(shap_values[0], list):
        shap_values = shap_values[0]

    shap_forcing = np.array(shap_values[0])
    shap_state = np.array(shap_values[1])

    shap_forcing_2d = shap_forcing.reshape(-1, len(forcing_cols))
    shap_state_2d = shap_state.reshape(-1, len(state_cols))

    test_f_2d = test_f.cpu().numpy().reshape(-1, len(forcing_cols))
    test_s_2d = test_s.cpu().numpy().reshape(-1, len(state_cols))

    shap_combined = np.concatenate([shap_forcing_2d, shap_state_2d], axis=1)
    features_combined = np.concatenate([test_f_2d, test_s_2d], axis=1)
    feature_names = forcing_cols + state_cols

    plt.figure(figsize=(12, 10))
    shap.summary_plot(shap_combined, features_combined, feature_names=feature_names, show=False)
    plt.title("SHAP Summary: Global Feature Impact on GPP (Time-Flattened)", fontname='Arial', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(output_img_folder, "SHAP_Summary_Plot.png"), dpi=300)
    plt.close()

    plt.figure(figsize=(12, 10))
    shap.summary_plot(shap_combined, features_combined, feature_names=feature_names, plot_type="bar", show=False)
    plt.title("SHAP Global Feature Importance (Magnitude)", fontname='Arial', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(output_img_folder, "SHAP_Bar_Plot.png"), dpi=300)
    plt.close()

    print(f"✅ SHAP 生成完成，保存至: {output_img_folder}")

# ==========================================
# 5. 主流程
# ==========================================
def train_time_aware_model():
    seq_len = 96
    batch_size = 64
    num_epochs = 100
    learning_rate = 0.001
    patience = 10
    TIME_COLUMN_NAME = 'date'
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    data_folder = r"C:\Users\Admin\Desktop\实验四\全波段全变量DT"
    output_img_folder = os.path.join(data_folder, "全149站点全局标准化森林类")
    os.makedirs(output_img_folder, exist_ok=True)

    forcing_cols = ['SW_IN_F', 'SW_IN_POT', 'CO2_F_MDS', 'P_F', 'VPD_F', 'TA_F', 'TS_F_MDS_1', 'SWC_F_MDS_1', 'WS_F']
    state_cols = ['EPIC_Available_Mask', 'Band317nm_Ref', 'Band325nm_Ref', 'Band340nm_Ref',
                  'Band388nm_Ref', 'Band443nm_Ref', 'Band551nm_Ref', 'Band680nm_Ref',
                  'Band688nm_Ref', 'Band764nm_Ref', 'Band780nm_Ref', 'Mean_SZA', 'Mean_VZA', 'Mean_RAA']
    static_cols = ['Lat', 'Long']
    lc_col = 'Veg_ID'

    # ==========================================
    # 在这里指定你的 训练集、验证集、测试集 站点名称
    # 只要 CSV 文件名包含列表中的字符串就会被划分入对应集合
    # ==========================================
    # 包含 DBF, EBF, ENF, MF/IMF
    train_sites = [
    'AU-Cum_EBF', 'AU-Gin_EBF', 'BE-Vie_MF', 'CH-Lae_MF', 'CN-Ash_EBF',
    'CN-BeO_MF', 'CN-HeM_DBF', 'CZ-Lnz_DBF', 'CZ-Stn_DBF', 'DE-Hai_DBF',
    'ES-Hen_ENF', 'FI-Ken_ENF', 'FR-Fon_DBF', 'JP-Tef_DNF', 'KR-HcM_MF',
    'KR-PcD_DBF', 'KR-WdE_EBF', 'NL-Loo_ENF', 'US-MOz_DBF', 'US-NC2_ENF',
    'US-Rpf_DBF', 'US-xDJ_ENF', 'US-xJE_ENF', 'US-xSB_ENF', 'US-xSE_DBF',
    'US-xWR_ENF'
]

    val_sites = [
    'US-CMW_DBF', 'US-xBL_DBF', 'US-UMd_DBF', 'IT-BsB_DBF', 'US-MMS_DBF',
    'AU-Sno_EBF', 'CA-TP3_ENF', 'RU-Fyo_ENF', 'US-CRK_ENF', 'RU-Fy2_ENF',
    'DE-Har_ENF', 'CA-TP4_ENF', 'KR-ScC_ENF', 'US-SP1_ENF', 'ES-HeB_ENF',
    'IT-SR2_ENF', 'IL-Yat_ENF', 'US-xUN_MF'
]

    test_sites = [
    'SE-Ros_ENF', 'FI-Var_ENF', 'KR-AdC_ENF', 'US-Me6_ENF', 'DE-Obe_ENF',
    'IT-Lav_ENF', 'BE-Bra_ENF', 'EE-Stg_ENF', 'FI-Hyy_ENF', 'CZ-RAJ_ENF',
    'US-MEF_ENF', 'DK-Sor_DBF', 'US-xBR_DBF', 'US-xUK_DBF', 'US-xTR_DBF',
    'AU-GWW_EBF', 'JP-Fhk_DNF', 'KR-JjM_MF'
]
    all_files = glob.glob(os.path.join(data_folder, "*.[cC][sS][vV]"))
    if not all_files:
        print("❌ 错误：未在指定目录找到CSV文件。")
        return

    # 严格按照给定的站点列表划分文件
    train_files, val_files, test_files = [], [], []
    unused_files = []

    for f in all_files:
        fname = os.path.basename(f)
        if any(site in fname for site in train_sites):
            train_files.append(f)
        elif any(site in fname for site in val_sites):
            val_files.append(f)
        elif any(site in fname for site in test_sites):
            test_files.append(f)
        else:
            unused_files.append(fname)

    print(f"📁 文件夹中共找到 {len(all_files)} 个站点文件。")
    print(f"   -> 划分为训练集: {len(train_files)} | 验证集: {len(val_files)} | 测试集: {len(test_files)}")

    if unused_files:
        print(f"⚠️ 注意: 有 {len(unused_files)} 个文件未被划分到任何集合，将被忽略。")
        # print("未分配的文件列表: ", unused_files) # 取消注释以查看未分配的文件名

    if not train_files:
        print("❌ 错误：训练集文件为空，请检查 train_sites 列表是否正确匹配了文件名。")
        return

    train_dataset = TimeAwareMultiStationDataset(
        train_files, seq_len, time_col=TIME_COLUMN_NAME,
        forcing_cols=forcing_cols, state_cols=state_cols,
        static_cols=static_cols, lc_col=lc_col, split_type='train'
    )

    # 提取训练集计算出的输入特征与目标变量的全局均值和标准差
    f_mean_f, f_std_f = train_dataset.feat_mean_f, train_dataset.feat_std_f
    f_mean_s, f_std_s = train_dataset.feat_mean_s, train_dataset.feat_std_s
    s_mean, s_std = train_dataset.static_mean, train_dataset.static_std
    t_mean, t_std = train_dataset.target_mean, train_dataset.target_std

    val_dataset = TimeAwareMultiStationDataset(
        val_files, seq_len, time_col=TIME_COLUMN_NAME,
        forcing_cols=forcing_cols, state_cols=state_cols,
        static_cols=static_cols, lc_col=lc_col,
        feat_mean_f=f_mean_f, feat_std_f=f_std_f,
        feat_mean_s=f_mean_s, feat_std_s=f_std_s,
        static_mean=s_mean, static_std=s_std,
        target_mean=t_mean, target_std=t_std, split_type='val'
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    model = TCN_Transformer_CrossAttention(
        num_forcing_features=len(forcing_cols),
        num_state_features=len(state_cols),
        seq_len=seq_len,
        num_static=len(static_cols),
        num_lc_classes=13,
        lc_embed_dim=8
    ).to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    checkpoint_latest_path = os.path.join(output_img_folder, "checkpoint_latest.pth")
    checkpoint_best_path = os.path.join(output_img_folder, "checkpoint_best.pth")

    start_epoch = 0
    best_val_loss = float('inf')
    epochs_no_improve = 0

    if os.path.exists(checkpoint_latest_path):
        print(f"\n🔄 检测到本地进度，尝试恢复...")
        try:
            checkpoint = torch.load(checkpoint_latest_path, map_location=device)
            model.load_state_dict(checkpoint['model_state_dict'])
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            start_epoch = checkpoint['epoch'] + 1
            best_val_loss = checkpoint['best_val_loss']
            epochs_no_improve = checkpoint['epochs_no_improve']
            print(f"✅ 从第 {start_epoch} 轮恢复，历史最佳 MSE: {best_val_loss:.4f}")
        except Exception as e:
            print(f"⚠️ 无法加载历史进度，将重新开始。错误信息: {e}")

    if start_epoch < num_epochs and epochs_no_improve < patience:
        print(f"🚀 开始训练...\n" + "-"*40)
        for epoch in range(start_epoch, num_epochs):
            model.train()
            train_loss = 0

            for batch_forcing, batch_state, batch_time, batch_static, batch_lc, batch_y, _ in train_loader:
                batch_forcing, batch_state, batch_time = batch_forcing.to(device), batch_state.to(device), batch_time.to(device)
                batch_static, batch_lc, batch_y = batch_static.to(device), batch_lc.to(device), batch_y.to(device)

                optimizer.zero_grad()
                outputs = model(batch_forcing, batch_state, batch_time, batch_static, batch_lc)
                loss = criterion(outputs, batch_y)
                loss.backward()

                # 梯度裁剪
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

                optimizer.step()
                train_loss += loss.item()

            model.eval()
            val_loss = 0
            with torch.no_grad():
                for batch_forcing, batch_state, batch_time, batch_static, batch_lc, batch_y, _ in val_loader:
                    batch_forcing, batch_state, batch_time = batch_forcing.to(device), batch_state.to(device), batch_time.to(device)
                    batch_static, batch_lc, batch_y = batch_static.to(device), batch_lc.to(device), batch_y.to(device)

                    outputs = model(batch_forcing, batch_state, batch_time, batch_static, batch_lc)
                    loss = criterion(outputs, batch_y)
                    val_loss += loss.item()

            avg_train_loss = train_loss / len(train_loader)
            avg_val_loss = val_loss / len(val_loader)
            print(f"Epoch [{epoch+1:03d}/{num_epochs}] | Train MSE (Std Scaled): {avg_train_loss:.4f} | Val MSE (Std Scaled): {avg_val_loss:.4f}")

            if avg_val_loss < best_val_loss:
                print(f"   🌟 新的最佳模型！MSE: {avg_val_loss:.4f}。")
                best_val_loss = avg_val_loss
                epochs_no_improve = 0
                torch.save(model.state_dict(), checkpoint_best_path)
            else:
                epochs_no_improve += 1
                print(f"   ⏳ 验证集未改善 (早停: {epochs_no_improve}/{patience})")

            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_val_loss': best_val_loss,
                'epochs_no_improve': epochs_no_improve
            }, checkpoint_latest_path)

            if epochs_no_improve >= patience:
                print(f"\n🛑 早停机制触发，训练结束。")
                break
            print("-" * 40)

    # ==========================================
    # 6. 分站点独立测试评估 (整合高级绘图逻辑)
    # ==========================================
    if os.path.exists(checkpoint_best_path):
        print(f"\n🎯 开始测试集评估...")
        model.load_state_dict(torch.load(checkpoint_best_path, map_location=device))
    model.eval()

    global_all_preds, global_all_targets = [], []
    for test_file in test_files:
        station_name = os.path.basename(test_file).replace('.csv', '')
        single_test_dataset = TimeAwareMultiStationDataset(
            [test_file], seq_len, time_col=TIME_COLUMN_NAME,
            forcing_cols=forcing_cols, state_cols=state_cols,
            static_cols=static_cols, lc_col=lc_col,
            feat_mean_f=f_mean_f, feat_std_f=f_std_f,
            feat_mean_s=f_mean_s, feat_std_s=f_std_s,
            static_mean=s_mean, static_std=s_std,
            target_mean=t_mean, target_std=t_std, split_type='test'
        )

        if len(single_test_dataset) == 0: continue

        single_test_loader = DataLoader(single_test_dataset, batch_size=batch_size, shuffle=False)
        all_preds, all_targets, all_times = [], [], []

        with torch.no_grad():
            for batch_forcing, batch_state, batch_time, batch_static, batch_lc, batch_y, batch_dt in single_test_loader:
                batch_forcing, batch_state, batch_time = batch_forcing.to(device), batch_state.to(device), batch_time.to(device)
                batch_static, batch_lc = batch_static.to(device), batch_lc.to(device)

                outputs = model(batch_forcing, batch_state, batch_time, batch_static, batch_lc)
                all_preds.extend(outputs.cpu().numpy())
                all_targets.extend(batch_y.numpy())
                all_times.extend(batch_dt)

        all_preds, all_targets = np.array(all_preds), np.array(all_targets)
        all_times = pd.to_datetime(all_times)

        # 反标准化：将模型输出的 Z-score 还原回原始的 GPP 物理刻度
        all_preds = all_preds * t_std + t_mean
        all_targets = all_targets * t_std + t_mean

        # 应用数据清洗要求：剔除 GPP < 0 的异常值
        valid_mask = all_targets >= 0
        plot_targets = all_targets[valid_mask]
        plot_preds = all_preds[valid_mask]
        plot_times = all_times[valid_mask]

        if len(plot_targets) == 0:
            continue

        global_all_preds.extend(plot_preds)
        global_all_targets.extend(plot_targets)

        station_mse = np.mean((plot_preds - plot_targets)**2)
        station_r2 = r2_score(plot_targets, plot_preds)

        print(f"📢 站点: {station_name} | 测试集 MSE: {station_mse:.4f} | 测试集 R²: {station_r2:.4f}")

        # 图表 1: 滑动平均趋势图
        window_size = 12
        all_targets_smooth = pd.Series(plot_targets).rolling(window=window_size, min_periods=1).mean().values
        all_preds_smooth = pd.Series(plot_preds).rolling(window=window_size, min_periods=1).mean().values

        plt.figure(figsize=(15, 6))
        plt.plot(plot_times, plot_targets, color='royalblue', linewidth=0.5, alpha=0.2, label='Actual (Raw)')
        plt.plot(plot_times, plot_preds, color='crimson', linewidth=0.5, alpha=0.2, label='Predicted (Raw)')
        plt.plot(plot_times, all_targets_smooth, label=f'Actual GPP (MA-{window_size})', color='royalblue', linewidth=1.5, alpha=0.9)
        plt.plot(plot_times, all_preds_smooth, label=f'Predicted GPP (MA-{window_size})', color='crimson', linewidth=1.5, linestyle='--', alpha=0.9)

        plt.title(f'[{station_name}] Test Set GPP Prediction (Moving Average Window={window_size})', fontsize=14, fontname='Arial')
        plt.xlabel('Time', fontsize=12, fontname='Arial')
        plt.ylabel('GPP Value', fontsize=12, fontname='Arial')
        plt.legend(prop={'family': 'Arial'})

        ax = plt.gca()
        ax.tick_params(direction='in')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        plt.grid(True, which='both', linestyle='--', alpha=0.5)
        plt.tight_layout()
        plt.savefig(os.path.join(output_img_folder, f"{station_name}_trend_ma.png"), dpi=300, facecolor='white', transparent=False)
        plt.close()

        # 图表 2: 局部放大图 (30天)
        zoom_days = 30
        steps_per_day = 24
        zoom_steps = zoom_days * steps_per_day
        zoom_steps = min(zoom_steps, len(plot_times))

        if zoom_steps > 0:
            peak_idx = np.argmax(all_targets_smooth)
            start_idx = max(0, peak_idx - zoom_steps // 2)
            end_idx = min(len(plot_times), start_idx + zoom_steps)

            if end_idx - start_idx < zoom_steps:
                start_idx = max(0, end_idx - zoom_steps)

            plt.figure(figsize=(15, 5))
            plt.plot(plot_times[start_idx:end_idx], plot_targets[start_idx:end_idx],
                     label='Actual GPP (Raw)', color='royalblue', linewidth=1.5)
            plt.plot(plot_times[start_idx:end_idx], plot_preds[start_idx:end_idx],
                     label='Predicted GPP (Raw)', color='crimson', linewidth=1.5, linestyle='--', alpha=0.8)

            peak_date_str = pd.to_datetime(plot_times.iloc[peak_idx] if isinstance(plot_times, pd.Series) else plot_times[peak_idx]).strftime('%Y-%m-%d')

            plt.title(f'[{station_name}] 30-Day Zoomed Prediction (Raw Data Detail around {peak_date_str})', fontsize=14, fontname='Arial')
            plt.xlabel('Time', fontsize=12, fontname='Arial')
            plt.ylabel('GPP Value', fontsize=12, fontname='Arial')
            plt.legend(prop={'family': 'Arial'})

            ax = plt.gca()
            ax.tick_params(direction='in')
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            plt.grid(True, which='both', linestyle='--', alpha=0.6)
            plt.xticks(rotation=30)
            plt.tight_layout()
            plt.savefig(os.path.join(output_img_folder, f"{station_name}_zoom_30days.png"), dpi=300, facecolor='white', transparent=False)
            plt.close()

        # 图表 3: 散点图
        plt.figure(figsize=(6, 6))
        plt.scatter(plot_targets, plot_preds, alpha=0.6, color='teal', s=15, edgecolors='k', linewidth=0.2)

        min_val = min(plot_targets.min(), plot_preds.min())
        max_val = max(plot_targets.max(), plot_preds.max())
        plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='1:1 Line')

        plt.title(f'[{station_name}] Actual vs Predicted Scatter', fontname='Arial')
        plt.xlabel('Actual GPP', fontname='Arial')
        plt.ylabel('Predicted GPP', fontname='Arial')
        plt.legend(prop={'family': 'Arial'})

        ax = plt.gca()
        ax.tick_params(direction='in')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        plt.tight_layout()
        plt.savefig(os.path.join(output_img_folder, f"{station_name}_scatter.png"), dpi=300, facecolor='white', transparent=False)
        plt.close()

        # ==========================================
        # 图表 4: 抽取任一年的数据展示时间序列
        # ==========================================
        years = plot_times.dt.year if hasattr(plot_times, 'dt') else pd.Series(plot_times).dt.year
        unique_years = years.unique()

        if len(unique_years) > 0:
            target_year = years.value_counts().idxmax()
            year_mask = (years == target_year).values

            y_times = plot_times[year_mask]
            y_targets = plot_targets[year_mask]
            y_preds = plot_preds[year_mask]

            plt.figure(figsize=(15, 5))
            plt.plot(y_times, y_targets, label=f'Actual GPP ({target_year})', color='royalblue', linewidth=1.2, alpha=0.8)
            plt.plot(y_times, y_preds, label=f'Predicted GPP ({target_year})', color='crimson', linewidth=1.2, linestyle='--', alpha=0.8)

            plt.title(f'[{station_name}] Full Year GPP Time Series ({target_year})', fontsize=14, fontname='Arial')
            plt.xlabel('Time', fontsize=12, fontname='Arial')
            plt.ylabel('GPP Value', fontsize=12, fontname='Arial')
            plt.legend(prop={'family': 'Arial'}, loc='upper right')

            ax = plt.gca()
            ax.tick_params(direction='in')
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            plt.grid(True, which='both', linestyle='--', alpha=0.4)
            plt.tight_layout()
            plt.savefig(os.path.join(output_img_folder, f"{station_name}_singleyear_{target_year}.png"), dpi=300, facecolor='white', transparent=False)
            plt.close()

    # ------------------------------------------
    # 7. 全局指标汇总与 SHAP 分析
    # ------------------------------------------
    if len(global_all_targets) > 0:
        global_all_preds = np.array(global_all_preds)
        global_all_targets = np.array(global_all_targets)

        global_mse = np.mean((global_all_preds - global_all_targets)**2)
        global_r2 = r2_score(global_all_targets, global_all_preds)

        print("\n" + "="*50)
        print("🌎 所有站点测试集全局评估结果")
        print("="*50)
        print(f"有效总测试样本数 (剔除 GPP<0): {len(global_all_targets)}")
        print(f"Global Test MSE: {global_mse:.4f}")
        print(f"Global Test R² Score: {global_r2:.4f}")
        print("="*50)

        try:
            perform_shap_analysis(model, val_loader, device, output_img_folder, forcing_cols, state_cols)
        except Exception as e:
            print(f"⚠️ SHAP 分析生成失败: {e}")

if __name__ == "__main__":
    train_time_aware_model()

📁 文件夹中共找到 149 个站点文件。
   -> 划分为训练集: 26 | 验证集: 18 | 测试集: 18
⚠️ 注意: 有 87 个文件未被划分到任何集合，将被忽略。
🚀 开始训练...
----------------------------------------
Epoch [001/100] | Train MSE (Std Scaled): 0.1283 | Val MSE (Std Scaled): 0.1963
   🌟 新的最佳模型！MSE: 0.1963。
----------------------------------------
Epoch [002/100] | Train MSE (Std Scaled): 0.0850 | Val MSE (Std Scaled): 0.1854
   🌟 新的最佳模型！MSE: 0.1854。
----------------------------------------
Epoch [003/100] | Train MSE (Std Scaled): 0.0731 | Val MSE (Std Scaled): 0.1813
   🌟 新的最佳模型！MSE: 0.1813。
----------------------------------------
Epoch [004/100] | Train MSE (Std Scaled): 0.0652 | Val MSE (Std Scaled): 0.1841
   ⏳ 验证集未改善 (早停: 1/10)
----------------------------------------
Epoch [005/100] | Train MSE (Std Scaled): 0.0595 | Val MSE (Std Scaled): 0.1880
   ⏳ 验证集未改善 (早停: 2/10)
----------------------------------------
Epoch [006/100] | Train MSE (Std Scaled): 0.0552 | Val MSE (Std Scaled): 0.1913
   ⏳ 验证集未改善 (早停: 3/10)
-------------------------------

C:\Users\admin\AppData\Local\Temp\ipykernel_8532\879165468.py:312: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_combined, features_combined, feature_names=feature_names, show=False)
C:\Users\admin\AppData\Local\Temp\ipykernel_8532\879165468.py:319: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_combined, features_combined, feature_names=feature_names, plot_type="bar", show=False)


✅ SHAP 生成完成，保存至: C:\Users\Admin\Desktop\实验四\全波段全变量DT\全149站点全局标准化森林类
